# 10 Point clouds for Potree

3D views of TNG300 for [Potree](https://github.com/potree/potree), written to `MyDrive/deepvoid_viz/subhalos_tweb/potree`:

- **`index.html`, whole box:** the TNG300-1 subhalo samples behind the training inputs, one cloud per spacing λ = 3, 5, 7, 10 h⁻¹ Mpc, each subhalo colored by the T-web class of its voxel.
- **`slab_L10.html`, one slab at λ = 10:** the subhalos in the slab; voxels the model gets wrong on the void/wall boundary; void surfaces from the T-web mask and from the prediction; and dark matter above a density cut, colored by log ρ/ρ̄. By default the slab is the run's test slab, the only part of the box not used in training or model selection.

Sections:
1. TNG300-1 subhalo catalog to Drive (once).
2. Check that the N most massive subhalos reproduce the gridded training inputs (optional after the first time).
3. Whole-box subhalo clouds.
4. Build PotreeConverter.
5. Whole-box page.
6. Full-box prediction at λ = 10 with a saved model (needs the GPU; saved to Drive, so later sessions skip it).
7. Slab clouds.
8. Slab page.

Needs Colab secrets `GITHUB_TOKEN`, and `TNG_API_KEY` only if the catalog isn't on Drive yet.

Runtime: **L4 GPU with High-RAM** for section 6; everything else runs on CPU with High-RAM. About 20 minutes for everything, less once the catalog and prediction are on Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
!pip -q install "laspy[lazrs]"
del token
!git -C /content/deepvoid log --oneline -1

## Settings

- `RUN_NAME`: the trained model for the slab view.
- `SLAB`: `'test'` for the run's test slab, `(axis, start, stop)` with start and stop in h⁻¹ Mpc for a slab of your choice, or `None` for the whole box (many millions of voxel points).
- `DENSITY_CUT`: dark-matter voxels with ρ/ρ̄ above this are drawn.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from deepvoid import Catalog
from deepvoid.labels import CLASS_NAMES

catalog = Catalog()
SIM = 'TNG300-1'
SPACINGS = (3, 5, 7, 10)
RUN_NAME = 'subhalo_C_mixed_smooth2ch_depth5'
SLAB_SPACING = 10
SLAB = 'test'
DENSITY_CUT = 3.0

subhalo_file = catalog.root / 'TNG' / f'{SIM}_subhalos_snap99.npz'
runs_root = Path('/content/drive/MyDrive/deepvoid_runs')
out = Path('/content/drive/MyDrive/deepvoid_viz/subhalos_tweb')
las_dir, page_dir = out / 'las', out / 'potree'

## 1. Subhalo catalog

About 14.5 million subhalos for TNG300-1. Skipped if the file is already on Drive.

In [ ]:
from deepvoid.sources.tng import fetch_subhalos, read_subhalos
from deepvoid.observe import select_by_spacing

if not subhalo_file.exists():
    fetch_subhalos(subhalo_file, api_key=userdata.get('TNG_API_KEY'), sim=SIM, tmp_dir='/content')
subhalos = read_subhalos(subhalo_file)
samples = {lam: select_by_spacing(subhalos, lam) for lam in SPACINGS}
print(f'{len(subhalos):,} subhalos from {subhalos.meta["sim"]}, saved at {subhalo_file}')

## 2. Check the selection against the training inputs

For each spacing, the N = (205 / λ)³ most massive subhalos are gridded with mass-weighted CIC on the 512³ grid and correlated with the field the models were trained on. A correlation of 1 means this is the same sample, gridded the same way. Both grid conventions are tried: grid points at integer multiples of the cell size (the package convention) and at cell centers.

Checked on 2026-11-13: correlation 1.0 with the integer grid at every spacing, so the training inputs are the top-N subhalos by `SubhaloMass`, and each subhalo belongs to the voxel of its nearest grid point. Rerun only if the catalog or the fields change.

In [ ]:
from deepvoid.pointcloud import compare_with_field

rows = {}
for lam, t in samples.items():
    field = catalog.load(f'tng300-3/subhalos/L{lam}').data
    rows[f'L{lam}'] = compare_with_field(t, field)
    del field
print(pd.DataFrame(rows).T.round(4).to_string())

## 3. Whole-box subhalo clouds

One LAS file per spacing: classification = T-web class of the subhalo's voxel (0 void, 1 wall, 2 filament, 3 halo), plus log₁₀ of the subhalo mass. The table compares the class fractions of the subhalos with those of all voxels.

In [ ]:
from deepvoid.pointcloud import values_at, write_las

mask = catalog.load('tng300-3/mask').data
fractions = {'all voxels': np.bincount(mask.ravel(), minlength=4) / mask.size}
for lam, t in samples.items():
    cls = values_at(mask, t.pos, t.box_size)
    write_las(las_dir / f'subhalos_L{lam}.las', t.pos, cls,
              extra={'log_mass': np.log10(t.mass).astype(np.float32)})
    fractions[f'L{lam} ({len(t):,} subhalos)'] = np.bincount(cls, minlength=4) / len(cls)
print('Fraction in each T-web class')
print(pd.DataFrame(fractions, index=CLASS_NAMES).T.round(3).to_string())

## 4. Build PotreeConverter

PotreeConverter 2.1.1 from source, a few minutes. The CMake warnings about old minimum versions and missing test files come from bundled libraries and are harmless.

In [ ]:
!apt-get -qq install -y libtbb-dev > /dev/null
!rm -rf /content/PotreeConverter
!git clone -q --depth 1 --branch 2.1.1 https://github.com/potree/PotreeConverter.git /content/PotreeConverter 2> /dev/null
!cmake -S /content/PotreeConverter -B /content/PotreeConverter/build -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1
!make -s -C /content/PotreeConverter/build -j8 2>&1 | tail -3
converter = '/content/PotreeConverter/build/PotreeConverter'
!{converter} --help | head -3

## 5. Whole-box page

λ = 3 shows first; the other spacings are under *Scene* in the sidebar.

In [ ]:
from deepvoid.pointcloud import build_page

clouds = [{'name': f'subhalos, λ = {lam}', 'las': las_dir / f'subhalos_L{lam}.las', 'visible': lam == SPACINGS[0]}
          for lam in SPACINGS]
print(build_page(converter, page_dir, clouds, title='TNG300-1 subhalos by T-web class'))

## 6. Prediction at λ = 10

Full-box prediction with the 32-cell margin between tiles. The predicted class is the class with the highest score (no tuned void threshold). Saved next to the visualization as the predicted class and the void score; skipped if already there.

In [ ]:
from deepvoid.runs import Run

run = Run(runs_root / RUN_NAME)
pred_file = out / 'predictions' / f'{RUN_NAME}_L{SLAB_SPACING}.npz'
if not pred_file.exists():
    scores = run.predict(f'tng300-3/subhalos/L{SLAB_SPACING}', margin=32)
    pred_file.parent.mkdir(parents=True, exist_ok=True)
    np.savez(pred_file, classes=scores.argmax(-1).astype(np.uint8),
             void_score=scores[..., 0].astype(np.float16))
    del scores
pred = np.load(pred_file)['classes']
truth = run.labels()
print('saved at', pred_file)

## 7. Slab clouds

- **Subhalos** in the slab, colored by T-web class.
- **Void/wall errors:** voxels that are wall in the T-web mask and predicted void, and the reverse.
- **Void surfaces:** void voxels with at least one non-void face neighbor, one-voxel shells around each void, from the T-web mask and from the prediction.
- **Dark matter:** voxels with ρ/ρ̄ above `DENSITY_CUT`, carrying log₁₀ ρ/ρ̄.

The printed fraction of slab wall voxels predicted void should be close to the test-slab value in notebook 06.

In [ ]:
from deepvoid.pointcloud import (DENSITY, SURFACE_PRED, SURFACE_TRUE, VOID_AS_WALL, WALL_AS_VOID,
                                 slab, void_surface, voxel_index, voxel_positions)

n, box = truth.shape[0], 205.0
if SLAB == 'test':
    r = run.regions['test']
    axis, start, stop = r.axis, r.start, r.stop
elif SLAB is None:
    axis, start, stop = 0, 0, n
else:
    axis, lo, hi = SLAB
    start, stop = round(lo / box * n), round(hi / box * n)
in_slab = slab(n, axis, start, stop)
cell_size = box / n
print(f'slab: axis {"xyz"[axis]}, {start * cell_size:.1f} to {stop * cell_size:.1f} h^-1 Mpc ({stop - start} cells)')

tag = f'L{SLAB_SPACING}_slab'
counts = {}

t = samples[SLAB_SPACING]
i = voxel_index(t.pos, n, box)[:, axis]
keep = (i >= start) & (i < stop)
write_las(las_dir / f'{tag}_subhalos.las', t.pos[keep], values_at(truth, t.pos[keep], box),
          extra={'log_mass': np.log10(t.mass[keep]).astype(np.float32)})
counts['subhalos'] = keep.sum()

wall_as_void = (truth == 1) & (pred == 0) & in_slab
void_as_wall = (truth == 0) & (pred == 1) & in_slab
pos = np.concatenate([voxel_positions(wall_as_void, box), voxel_positions(void_as_wall, box)])
codes = np.repeat([WALL_AS_VOID, VOID_AS_WALL], [wall_as_void.sum(), void_as_wall.sum()])
write_las(las_dir / f'{tag}_errors.las', pos, codes)
counts['true wall, predicted void'] = wall_as_void.sum()
counts['true void, predicted wall'] = void_as_wall.sum()
print(f'slab wall voxels predicted void: {wall_as_void.sum() / ((truth == 1) & in_slab).sum():.3f}')

for name, classes, code in (('surface_true', truth, SURFACE_TRUE), ('surface_pred', pred, SURFACE_PRED)):
    s = void_surface(classes) & in_slab
    write_las(las_dir / f'{tag}_{name}.las', voxel_positions(s, box), np.full(s.sum(), code))
    counts[name.replace('_', ', ')] = s.sum()

dm = catalog.load('tng300-3/dm').data
rho = dm / dm.mean()
dense = (rho > DENSITY_CUT) & in_slab
write_las(las_dir / f'{tag}_density.las', voxel_positions(dense, box), np.full(dense.sum(), DENSITY),
          extra={'log_density': np.log10(rho[dense]).astype(np.float32)})
counts[f'dark matter, rho/rho_bar > {DENSITY_CUT:g}'] = dense.sum()
del dm, rho

print(pd.Series(counts, name='points').to_string())

## 8. Slab page

At start-up: the subhalos, the void/wall errors and the T-web void surfaces. The predicted void surfaces and the dark matter are listed under *Scene*, switched off.

In [ ]:
log_rho_max = float(np.log10(DENSITY_CUT)) + 2.5
clouds = [
    {'name': f'subhalos, λ = {SLAB_SPACING}', 'las': las_dir / f'{tag}_subhalos.las', 'size': 6, 'visible': True},
    {'name': 'void/wall errors', 'las': las_dir / f'{tag}_errors.las', 'size': 2, 'visible': True},
    {'name': 'void surfaces, T-web', 'las': las_dir / f'{tag}_surface_true.las', 'size': 2, 'visible': True},
    {'name': 'void surfaces, predicted', 'las': las_dir / f'{tag}_surface_pred.las', 'size': 2, 'visible': False},
    {'name': f'dark matter, ρ/ρ̄ > {DENSITY_CUT:g}', 'las': las_dir / f'{tag}_density.las', 'size': 2,
     'visible': False, 'attribute': 'log_density', 'gradient': 'INFERNO',
     'range': [float(np.log10(DENSITY_CUT)), log_rho_max]},
]
print(build_page(converter, page_dir, clouds, title=f'TNG300 slab, λ = {SLAB_SPACING}, {RUN_NAME}',
                 page_name=f'slab_L{SLAB_SPACING}'))

## Viewing

Browsers won't load Potree pages straight from disk, so serve the folder locally:

1. Download `MyDrive/deepvoid_viz/subhalos_tweb/potree` and unzip it (`unzip` in Terminal if double-clicking fails).
2. In a terminal, in that folder: `python3 serve.py`. Use this rather than `python3 -m http.server`: the default server refuses most of the many requests Potree makes at once, which leaves parts of a cloud unloaded.
3. Open http://localhost:8000/index.html or http://localhost:8000/slab_L10.html

In the sidebar:
- **Scene** switches clouds on and off; each cloud's settings change point size and coloring.
- **Filters → Classification** hides or shows each kind of point.
- **Tools → Clip** cuts out boxes.